# NN Architecture 2C: Bidirectional LSTM for Rayleigh Fading

**Reference**: "Classification of Stochastic Systems with Deep Learning and Hypothesis Testing" (project paper)

**Approach**: LSTM networks capture temporal dependencies in stochastic processes, ideal for Rayleigh fading channels

**Rationale**: 
- Rayleigh fading channel is a stochastic process (time-varying gain)
- LSTM can learn dynamic relationships: $h(t) = f(h(t-1), \text{channel state})$
- Bidirectional LSTM uses both forward and backward information
- Gating mechanism (forget, input, output gates) learns what to remember

**Architecture**:
```
Input: [τ_correlator, h_est, SNR, energy] → reshape as sequence (4D time steps)
    ↓
Bidirectional LSTM(64, return_sequences=True) → Dropout(0.3)
    ↓
Bidirectional LSTM(32) → Dropout(0.2)
    ↓
Dense(32, ReLU)
    ↓
Dense(1, Sigmoid) → Binary output
```

**Framework**: PyTorch with LSTM cells

In [ ]:
# ==============================================================================
# LSTM FOR RAYLEIGH FADING CHANNELS
# Reference: Classification of Stochastic Systems with Deep Learning
# ==============================================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)
import h5py
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load dataset
with h5py.File('dataset_nn_100k.h5', 'r') as f:
    X_train = torch.FloatTensor(f['X_train'][:])
    y_train = torch.FloatTensor(f['y_train'][:])
    X_val = torch.FloatTensor(f['X_val'][:])
    y_val = torch.FloatTensor(f['y_val'][:])
    X_test = torch.FloatTensor(f['X_test'][:])
    y_test = torch.FloatTensor(f['y_test'][:])

# Reshape for LSTM: treat features as sequence
# (N, 4) → (N, 4, 1) for 4 features as temporal steps
X_train_lstm = X_train.unsqueeze(2)  # (N, 4, 1)
X_val_lstm = X_val.unsqueeze(2)
X_test_lstm = X_test.unsqueeze(2)

# DataLoaders
train_loader = DataLoader(
    TensorDataset(X_train_lstm, y_train.unsqueeze(1)),
    batch_size=256, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(X_val_lstm, y_val.unsqueeze(1)),
    batch_size=256
)
test_loader = DataLoader(
    TensorDataset(X_test_lstm, y_test.unsqueeze(1)),
    batch_size=256
)

# Define LSTM model
class BiLSTM_Rayleigh(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, dropout=0.3):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(dropout)
        
        self.lstm2 = nn.LSTM(hidden_size*2, hidden_size//2, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(dropout)
        
        self.fc1 = nn.Linear(hidden_size, 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # x: (batch, seq_len=4, input_size=1)
        lstm_out, _ = self.lstm1(x)  # (batch, seq_len, hidden*2)
        lstm_out = self.dropout1(lstm_out)
        
        lstm_out, _ = self.lstm2(lstm_out)  # (batch, seq_len, hidden)
        lstm_out = self.dropout2(lstm_out)
        
        # Use last output
        last_output = lstm_out[:, -1, :]  # (batch, hidden)
        
        x = torch.relu(self.fc1(last_output))
        x = self.fc2(x)
        x = self.sigmoid(x)
        
        return x

model = BiLSTM_Rayleigh(input_size=1, hidden_size=64, dropout=0.3).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

print(f"✓ BiLSTM Model created")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ==============================================================================
# TRAIN & EVALUATE
# ==============================================================================

def train_step(model, loader, criterion, optimizer, device):
    model.train()
    loss_total = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        loss_total += loss.item() * X_batch.size(0)
    return loss_total / len(loader.dataset)

def eval_step(model, loader, criterion, device):
    model.eval()
    loss_total = 0
    all_pred = []
    all_true = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss_total += loss.item() * X_batch.size(0)
            all_pred.extend(y_pred.cpu().numpy())
            all_true.extend(y_batch.cpu().numpy())
    return loss_total / len(loader.dataset), np.array(all_pred), np.array(all_true)

# Training loop
num_epochs = 100
best_val_loss = float('inf')
patience_count = 0
history = {'train': [], 'val': []}

print("Training BiLSTM...")
for epoch in range(num_epochs):
    train_loss = train_step(model, train_loader, criterion, optimizer, device)
    val_loss, _, _ = eval_step(model, val_loader, criterion, device)
    
    history['train'].append(train_loss)
    history['val'].append(val_loss)
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}: Train={train_loss:.4f}, Val={val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_count = 0
        torch.save(model.state_dict(), 'model_lstm_best.pth')
    else:
        patience_count += 1
        if patience_count >= 10:
            print(f"Early stopping at epoch {epoch+1}")
            break

print("✓ Training complete")

# Evaluate
model.load_state_dict(torch.load('model_lstm_best.pth'))
_, y_train_pred, y_train_true = eval_step(model, train_loader, criterion, device)
_, y_val_pred, y_val_true = eval_step(model, val_loader, criterion, device)
_, y_test_pred, y_test_true = eval_step(model, test_loader, criterion, device)

def show_metrics(y_true, y_pred, name):
    y_pred_bin = (y_pred > 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred_bin)
    fnr = confusion_matrix(y_true, y_pred_bin).ravel()[2] / (confusion_matrix(y_true, y_pred_bin).ravel()[2] + confusion_matrix(y_true, y_pred_bin).ravel()[3]) if len(confusion_matrix(y_true, y_pred_bin).ravel()) == 4 else 0
    auc = roc_auc_score(y_true, y_pred)
    print(f"{name}: Acc={acc:.4f}, AUC={auc:.4f}, FNR={fnr:.6f}")

show_metrics(y_train_true, y_train_pred, "Train")
show_metrics(y_val_true, y_val_pred, "Val")
show_metrics(y_test_true, y_test_pred, "Test")

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train'], label='Train')
ax.plot(history['val'], label='Val')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('BiLSTM Training (Rayleigh Fading)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results_lstm.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ LSTM training complete!")